# Відповіді на запитання відкритого типу (ODQA або Open-domain question answering)

У цій лабораторній роботі Ви познайомитеся із задачею відповіді на запитання відкритого типу. На відміну від стандартної задачі відповіді на запитання, для якої потребується і запитання і контекст з можливою відповіддю на поставлене запитання, відповідь на запитання відкритого типу потребує лише запитання.
Задача ODQA є досить поширеною, а однією з її варіацій є функція Google Quick (Direct) Answer, що дозволяє отримувати швидкі відповіді на деякі запити прямо на сторінці пошукової видачі ще до переходу на інтернет-ресурс.

![Google Quick Answer](https://raw.githubusercontent.com/niksyromyatnikov/opnu-ml-assignments/refs/heads/niksyromyatnikov/docs/readme/2026-open-domain-question-answering/google-quick-answer.png)

## Архітектура

Існує декілька основних підходів до побудови ODQA-систем, дізнатися більше про які можна за [посиланням](https://lilianweng.github.io/posts/2020-10-29-odqa/).
У рамках цієї роботи Ви побудуєте двоетапне рішення, що складається з пошуковика (**Retriever**) та читача (**Reader**).
- **Retriever** для наданого запиту (запитання) "витягує" зі сховища даних з тисячами чи мільйонами записів релевантні документи, що найбільш ймовірно відповідають на поставлене запитання.
- **Reader** у обраних релевантних документах намагається знайти сегменти (слова, словосполучення або речення), що містять повну відповідь на запитання.

<center>
    <figure>
        <img src="https://lilianweng.github.io/posts/2020-10-29-odqa/QA-retriever-reader.png" alt ="Concrete">
        <figcaption>
            <a href="https://lilianweng.github.io/posts/2020-10-29-odqa/QA-retriever-reader.png">How to Build an Open-Domain Question Answering System? by Lilian Weng</a></figcaption>
    </figure>
</center>

## Підготовка середовища

Процедури тренування моделей потребують значних обчислювальних ресурсів, тому для виконання лабораторної роботи бажано використовувати сервіси з доступом до GPU (Kaggle/Google Colab).

In [ ]:
# виводить інформацію про графічний прискорювач та його поточне навантаження
!nvidia-smi

Wed Nov 18 01:37:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%pip uninstall -y torch torchvision torchaudio torchtext xformers

%pip install --no-cache-dir --force-reinstall \
  torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.14.1+cu130
Uninstalling torch-2.14.1+cu130:
  Successfully uninstalled torch-2.14.1+cu130
Found existing installation: torchvision 0.29.1+cu130
Uninstalling torchvision-0.29.1+cu130:
  Successfully uninstalled torchvision-0.29.1+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 126.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 341.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 69.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 147.1 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 128.9 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 136.9 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

### Перезавантажити середовище: Run -> Restart & clear cell outputs

In [ ]:
# Встановлення залежностей
%pip install -q \
    "transformers==5.8.0" \
    "datasets==4.8.5" \
    "evaluate==0.4.6" \
    "accelerate==1.13.0" \
    "huggingface-hub==1.14.0" \
    "sentence-transformers==5.4.1" \
    "langchain==1.2.18" \
    "langchain-community==0.4.1" \
    "langchain-huggingface==1.2.2" \
    "faiss-cpu==1.13.2"

%pip uninstall -y torchao torchcodec

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 1.14.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.2.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 1.14.0 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 1.14.0 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Found existing installation: torchcodec 0.17.0+cu130
Uninstalling torchcodec-0.

In [ ]:
!python --version

Python 3.13.16


### Знову перезавантажити середовище: Run -> Restart & clear cell outputs

In [ ]:
# Імпорт бібліотек
import random
import re
from itertools import combinations
from collections import Counter

import torch
from transformers import AutoTokenizer, DefaultDataCollator, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from datasets import load_dataset, Dataset, DatasetDict
from evaluate import load
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from huggingface_hub import login

In [ ]:
# визначення обчислювального пристрою для роботи, cuda використовується для сеансу з обраним графічним прискорювачем
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## 1. Підготовка даних

У лабораторній роботі використовується [частково локалізована версія датасету SQuAD 2.0](https://huggingface.co/datasets/FIdo-AI/ua-squad), що містить тексти, запитання, а також розмічені сегменти у текстах, що є відповідями на запитання. Друга версія датасету відрізняється від першої тим, що деякі запитання можуть не мати відповіді.

In [ ]:
train_raw = load_dataset(
    "json",
    data_files="hf://datasets/FIdo-AI/ua-squad/train.json",
    split="train",
)

train.json:   0%|          | 0.00/16.9M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def normalize_train(split):
    rows = []
    skipped = 0
    no_answer = 0

    for i, ex in enumerate(split):
        item = ex["data"]

        question = item["question"]
        context = item["context"]
        answer = item.get("answer", None)

        if answer is None or str(answer).strip() == "":
            no_answer += 1
            rows.append({
                "id": f"train-{i}",
                "question": question,
                "context": context,
                "answers": {
                    "text": [],
                    "answer_start": [],
                },
                "is_impossible": True,
            })
            continue

        answer_start = context.find(answer)

        if answer_start == -1:
            skipped += 1
            continue

        rows.append({
            "id": f"train-{i}",
            "question": question,
            "context": context,
            "answers": {
                "text": [answer],
                "answer_start": [answer_start],
            },
            "is_impossible": False,
        })

    print(f"No-answer train examples kept: {no_answer}")
    print(f"Skipped train examples with invalid non-empty span: {skipped}")

    return Dataset.from_list(rows)

train_dataset = normalize_train(train_raw)

No-answer train examples kept: 2927
Skipped train examples with invalid non-empty span: 0


In [ ]:
# детермінованість
random.seed(42)

In [ ]:
train_dataset[0]

{'id': 'train-0',
 'question': 'Якою була мета нової творчої компанії DONDA, створеної Каньє?',
 'context': '5 січня 2012 року Вест оголосив про створення компанії для творчого контенту DONDA, названої на честь його покійної матері Донди Вест. Під час представлення Вест заявив, що компанія "продовжить там, де зупинився Стів Джобс"; DONDA діятиме як "дизайнерська компанія, яка забезпечить мислителям творчий простір для реалізації своїх мрій та ідей" з "метою виготовлення продуктів та поширення досвіду, які люди хочуть отримати й можуть собі дозволити". Вест, як відомо, мало говорить про діяльність компанії, відсутні як офіційний веб-сайт, так і представлення в соціальних мережах. Креативна філософія DONDA містить необхідність "розміщувати творців у спільному просторі разом із подібними думками", щоб "спростити та естетично вдосконалити все, що ми бачимо, смакуємо, торкаємось та відчуваємо". Сучасні критики відзначають незмінну мінімалістичну естетику, яка повторюється в багатьох творчих

In [ ]:
train_dataset = DatasetDict({
    "train": train_dataset
})

train_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'answers', 'is_impossible'],
        num_rows: 13859
    })
})

In [ ]:
# поділ датасету на тренувальний, валідаційний та тестовий набори
# для запобігання витоку тренувальних даних у валідаційний та тестовий набори
# необхідно ділити саме за контекстами, адже кожний текст може мати декілька запитань
def generate_split(qa_dataset):
    contexts = list(set(qa_dataset["train"]["context"]))
    contexts.sort()

    random.shuffle(contexts)

    n = len(contexts)
    train_end = int(n * 0.8)
    val_end = int(n * 0.9)

    train_contexts = set(contexts[:train_end])
    val_contexts = set(contexts[train_end:val_end])
    test_contexts = set(contexts[val_end:])

    print(len(train_contexts), len(val_contexts), len(test_contexts))

    train_rows = qa_dataset["train"].filter(
        lambda x: x["context"] in train_contexts
    )

    val_rows = qa_dataset["train"].filter(
        lambda x: x["context"] in val_contexts
    )

    test_rows = qa_dataset["train"].filter(
        lambda x: x["context"] in test_contexts
    )

    return train_rows, val_rows, test_rows

train_rows, val_rows, test_rows = generate_split(train_dataset)
len(train_rows), len(val_rows), len(test_rows)

2096 262 262


Filter:   0%|          | 0/13859 [00:00<?, ? examples/s]

Filter:   0%|          | 0/13859 [00:00<?, ? examples/s]

Filter:   0%|          | 0/13859 [00:00<?, ? examples/s]

(11080, 1339, 1440)

In [ ]:
train_rows[200:205]

{'id': ['train-254', 'train-255', 'train-256', 'train-257', 'train-258'],
 'question': ['Що зробив Шуддгодана, щоб його син став королем, а не святою людиною?',
  'Скільки років було Гаутамі, коли він вперше вийшов з палацу?',
  'Що сталося, коли Гаутама вийшов з палацу?',
  'На кого натрапив Будда у чотирьох визначих зустрічах буддизму?',
  'Яким батько Будди хотів, щоб він став?'],
 'context': ['Шуддгодана бажав бачити свого сина королем, тому заборонив йому виходити з палацу. Але в 29 років, незважаючи на зусилля батька, Гаутама все ж кілька разів полишав палац. Завдяки кільком зустрічам, відомим у буддистській літературі як Чотири зустрічі, він дізнався про страждання простих людей, натрапивши на старого, хворого, труп і, нарешті, святого чоловіка, що був, вочевидь, задоволеним світом і в гармонії з ним. Цей досвід надихнув Гаутаму відмовитись від королівського життя й розпочати духовні пошуки.',
  'Шуддгодана бажав бачити свого сина королем, тому заборонив йому виходити з палацу. 

In [ ]:
def normalize_text_for_check(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def make_key(example, fields):
    return tuple(normalize_text_for_check(example[field]) for field in fields)


def check_split_leakage(splits, fields=("context",), max_examples=3, raise_error=True):
    print(f"Checking leakage by fields: {fields}")

    split_keys = {
        split_name: set(make_key(example, fields) for example in dataset)
        for split_name, dataset in splits.items()
    }

    has_leakage = False

    for left_name, right_name in combinations(split_keys.keys(), 2):
        overlap = split_keys[left_name] & split_keys[right_name]

        print(f"{left_name} <-> {right_name}: {len(overlap)} overlapping keys")

        if overlap:
            has_leakage = True
            print("Examples of overlapping keys:")
            for item in list(overlap)[:max_examples]:
                print(item)
            print()

    if has_leakage and raise_error:
        raise ValueError(f"Data leakage detected for fields: {fields}")

    return not has_leakage


def check_internal_duplicates(splits, fields=("question", "context"), max_examples=3):
    print(f"Checking internal duplicates by fields: {fields}")

    duplicate_report = {}

    for split_name, dataset in splits.items():
        keys = [make_key(example, fields) for example in dataset]
        counts = Counter(keys)

        duplicates = {
            key: count
            for key, count in counts.items()
            if count > 1
        }

        duplicate_report[split_name] = duplicates

        duplicate_count = sum(count - 1 for count in duplicates.values())

        print(f"{split_name}: {duplicate_count} duplicated rows")

        if duplicates:
            print("Examples of duplicates:")
            for key, count in list(duplicates.items())[:max_examples]:
                print(f"{count}x:", key)
            print()

    return duplicate_report

In [ ]:
splits = {
    "train": train_rows,
    "validation": val_rows,
    "test": test_rows,
}

# Контекст не має бути одночасно у train, val, test
check_split_leakage(
    splits,
    fields=("context",),
    raise_error=True,
)

check_split_leakage(
    splits,
    fields=("question", "context"),
    raise_error=True,
)

# Пошук дублікатів
duplicate_report = check_internal_duplicates(
    splits,
    fields=("question", "context"),
)

Checking leakage by fields: ('context',)
train <-> validation: 0 overlapping keys
train <-> test: 0 overlapping keys
validation <-> test: 0 overlapping keys
Checking leakage by fields: ('question', 'context')
train <-> validation: 0 overlapping keys
train <-> test: 0 overlapping keys
validation <-> test: 0 overlapping keys
Checking internal duplicates by fields: ('question', 'context')
train: 28 duplicated rows
Examples of duplicates:
2x: ('яким типом релігії є буддизм?', 'буддизм - це нетеїстична релігія або філософія (санскрит: धर्म dharma; палі: धम्म dhamma), що охоплює різноманітні традиції, вірування та духовні практики, в основному засновані на вченнях, приписуваних будді гаутама, широко відомому як будда ("пробуджений"). згідно з буддистською традицією, будда жив і навчав у східній частині індійського субконтиненту, сучасному непалі, десь між 6 і 4 століттями до н.е. буддисти вважають його пробудженим або просвітленим учителем, який ділився своєю мудрістю, щоб допомогти розумним

## 2. Побудова Reader-моделі

Для вирішення задачі відповіді на запитання за контекстом буде використано мережу на базі архітектури кодуючого трансформера, принцип роботи якої полягає у наступному:
- мережа отримує вхідну послідовність, що представляє собою запитання та контекст, розділені спеціальним токеном
- мережа кодує отриману послідовність
- доданий лінійний шар виконує декодування закодованої інформації шляхом оцінки імовірностей бути початком або кінцем сегменту відповіді для кожного токена (слова) з контексту

Фактично, задача визначення сегменту у послідовності дуже схожа на стандартну задачу класифікації, але тут класифікується не уся послідовність, а кожний її елемент.

[Детальніше про кодуючі трансформери](https://jalammar.github.io/illustrated-bert/)

[Детальніше про вирішення задачі класифікації за допомогою кодуючих трансформерів](https://jalammar.github.io/a-visual-guide-to-using-bert-for-the-first-time/)

<center>
    <figure>
        <img src="https://mccormickml.com/assets/BERT/SQuAD/input_formatting.png?ref=blog.paperspace.com" alt ="Concrete">
        <figcaption>
            <a href="https://mccormickml.com/assets/BERT/SQuAD/input_formatting.png?ref=blog.paperspace.com">A Comparison of Question Answering Models by Abhijith Neil Abraham</a></figcaption>
    </figure>
</center>

<center>
    <figure>
        <img src="https://blog.paperspace.com/content/images/size/w1000/2020/08/BERT.jpg" alt ="Concrete">
        <figcaption>
            <a href="https://blog.paperspace.com/content/images/size/w1000/2020/08/BERT.jpg">BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding by Jacob Devlin</a></figcaption>
    </figure>
</center>

#### Варіанти для тренування моделі

Варіант = Номер_у_списку % 2 + 1

|**№**| Model                      | Epochs | Learning rate |
|-----|----------------------------|--------|---------------|
|**1**| microsoft/mdeberta-v3-base | 4      | 2e-05         |
|**2**| microsoft/mdeberta-v3-base | 3      | 3e-05         |

In [ ]:
# базова модель для тренування Reader
base_model_name = "microsoft/mdeberta-v3-base"

# натренована модель
tuned_model_name = "NLPForUA/mdeberta-v3-ua-squad-reader"

# True - тренувати базову модель
# False - пропустити тренування та завантажити вже натреновану модель
TRAIN_MODEL = True

# Під час тренування токенізатор береться з базової моделі.
# Під час повторного запуску - з репозиторію вже натренованої моделі.
model_name = base_model_name if TRAIN_MODEL else tuned_model_name
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

In [ ]:
#### (Опціонально) Збереження Reader-моделі на Hugging Face Hub

# Після тренування модель і токенізатор можна завантажити у власний репозиторій на Hugging Face Hub (https://huggingface.co/). Це дозволяє не повторювати тривале тренування.
# Для завантаження моделі на Hub необхідна авторизація. Токен Hugging Face не потрібно записувати безпосередньо у ноутбук: функція `login()` запитає його під час виконання.

# Репозиторій, у який буде завантажено натреновану модель.
# Замініть YOUR_HF_USERNAME на власний логін Hugging Face.
PUBLISH_MODEL = False
HF_MODEL_ID = "{YOUR_HF_USERNAME}/mdeberta-v3-ua-squad-reader"

if PUBLISH_MODEL:
    login()

In [ ]:
# опис параметрів https://github.com/huggingface/transformers/blob/v4.44.0/examples/pytorch/question-answering/run_qa.py#L105-L206

question_column_name = "question"
context_column_name = "context"
answer_column_name = "answers"
max_seq_length = 384
doc_stride = 128
pad_to_max_length = True
pad_on_right = tokenizer.padding_side == "right"

# підготовка даних для тренування
def prepare_train_features(examples):
    # У деяких питаннях зліва багато пробілів, що може спричинити помилки обрізання послідовностей,
    # тому ми їх видаляємо
    examples[question_column_name] = [q.lstrip() for q in examples[question_column_name]]

    tokenized_examples = tokenizer(
        examples[question_column_name if pad_on_right else context_column_name],
        examples[context_column_name if pad_on_right else question_column_name],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_seq_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length" if pad_to_max_length else False,
    )

    # Оскільки один приклад з довгим контекстом може дати нам кілька ознак,
    # нам потрібно мати зв'язок між ознакою та відповідним прикладом.
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        # We will label impossible answers with the index of the CLS token.
        input_ids = tokenized_examples["input_ids"][i]
        if tokenizer.cls_token_id in input_ids:
            cls_index = input_ids.index(tokenizer.cls_token_id)
        elif tokenizer.bos_token_id in input_ids:
            cls_index = input_ids.index(tokenizer.bos_token_id)
        else:
            cls_index = 0

        # Отримуємо послідовність, що відповідає цьому прикладу
        sequence_ids = tokenized_examples.sequence_ids(i)

        # Один приклад може мати кілька сегментів
        # отримуємо індекс, що містить цей проміжок тексту.
        sample_index = sample_mapping[i]
        answers = examples[answer_column_name][sample_index]
        # Якщо запитання не має відповіді, то використовуємо спеціальний токен
        # у якості правильної відповіді
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # Індекс початку та кінця відповіді в тексті
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Знаходження індексу початку поточного сегменту
            token_start_index = 0
            while sequence_ids[token_start_index] != (1 if pad_on_right else 0):
                token_start_index += 1

            # Знаходження індексу кінця поточного сегменту
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != (1 if pad_on_right else 0):
                token_end_index -= 1

            # Визначення того, чи входить розмічена відповідь у поточний сегмент
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

# підготовка даних для тестування
def prepare_validation_features(examples):
    examples[question_column_name] = [q.lstrip() for q in examples[question_column_name]]

    tokenized_examples = tokenizer(
        examples[question_column_name if pad_on_right else context_column_name],
        examples[context_column_name if pad_on_right else question_column_name],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_seq_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length" if pad_to_max_length else False,
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")

    tokenized_examples["example_id"] = []

    for i in range(len(tokenized_examples["input_ids"])):
        sequence_ids = tokenized_examples.sequence_ids(i)
        context_index = 1 if pad_on_right else 0

        sample_index = sample_mapping[i]
        tokenized_examples["example_id"].append(examples["id"][sample_index])

        tokenized_examples["offset_mapping"][i] = [
            (o if sequence_ids[k] == context_index else None)
            for k, o in enumerate(tokenized_examples["offset_mapping"][i])
        ]

    return tokenized_examples

In [ ]:
train_set = Dataset.from_list(train_rows)
val_set = Dataset.from_list(val_rows)
test_set = Dataset.from_list(test_rows)
train_set, val_set, test_set

(Dataset({
     features: ['id', 'question', 'context', 'answers', 'is_impossible'],
     num_rows: 11080
 }),
 Dataset({
     features: ['id', 'question', 'context', 'answers', 'is_impossible'],
     num_rows: 1339
 }),
 Dataset({
     features: ['id', 'question', 'context', 'answers', 'is_impossible'],
     num_rows: 1440
 }))

In [ ]:
train_set[0]

In [ ]:
tokenized_train = None
tokenized_val = None
tokenized_test = test_set.map(prepare_validation_features, batched=True, remove_columns=test_set.column_names)

# підготовка даних для тренування та тестування
# Тренувальний і валідаційний набори потрібні лише тоді, коли модель навчається.
if TRAIN_MODEL:
    tokenized_train = train_set.map(prepare_train_features, batched=True, remove_columns=train_set.column_names)
    tokenized_val = val_set.map(prepare_train_features, batched=True, remove_columns=val_set.column_names)

Map:   0%|          | 0/1440 [00:00<?, ? examples/s]

Map:   0%|          | 0/11080 [00:00<?, ? examples/s]

Map:   0%|          | 0/1339 [00:00<?, ? examples/s]

In [ ]:
# завантаження моделі
model = AutoModelForQuestionAnswering.from_pretrained(model_name, dtype=torch.float32)
# розміщення моделі на графічному прискорювачі
model.to(device)

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForQuestionAnswering LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias          

DebertaV2ForQuestionAnswering(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(251000, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm(

In [ ]:
def check_model_for_nan(model):
    bad = []

    for name, param in model.named_parameters():
        if torch.isnan(param).any():
            bad.append(name)

    print("Parameters with NaN:", len(bad))

    for name in bad[:20]:
        print(name)

check_model_for_nan(model)

Parameters with NaN: 0


In [ ]:
if TRAIN_MODEL:
    # конфігурація тренера
    training_args = TrainingArguments(
        output_dir="deberta_squad_ukr_3epoch_16bs_3e05",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=16, # розмір пакету для тренування
        per_device_eval_batch_size=16, # розмір пакету для валідації
        num_train_epochs=3,
        weight_decay=0.01, # L2 регуляризація

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        data_collator=DefaultDataCollator(),
    )

    # тренування
    # важливо: для запуску тренування у Kaggle необхідно виконати інструкції wandb,
    # що будуть виведені на екран після запуску команди
    trainer.train()

else:
    print(f"Тренування пропущено. Використовується готова модель: {model_name}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss
1,1.952508,1.110733
2,1.115901,1.029526
3,0.631941,1.083357


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
if PUBLISH_MODEL and TRAIN_MODEL:
    # публікація натренованої Reader-моделі та токенізатора на Hugging Face Hub
    model.push_to_hub(HF_MODEL_ID)
    tokenizer.push_to_hub(HF_MODEL_ID)

    print(f"Модель і токенізатор завантажено на Hugging Face Hub: {HF_MODEL_ID}")
    print("Для повторного запуску встановіть TRAIN_MODEL = False.")

In [ ]:
# Переводимо модель у режим передбачення
model.eval()

DebertaV2ForQuestionAnswering(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(251000, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm(

In [ ]:
def answer_question(question, context, model, tokenizer, max_length=384):
    # Токенізуємо пару "питання + контекст".
    # truncation="only_second" означає, що при перевищенні max_length буде обрізатися саме контекст, а не питання.
    # return_offsets_mapping=True потрібен для відновлення текстової відповіді за позиціями токенів у початковому контексті.
    encoded = tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=max_length,
        return_offsets_mapping=True,
        return_tensors="pt",
    )

    # offset_mapping показує, яким символам у початковому тексті відповідає кожен токен.
    offset_mapping = encoded.pop("offset_mapping")[0].tolist()
    # sequence_ids показує, до якої частини входу належить кожен токен:
    # None - спеціальні токени, наприклад CLS/SEP;
    # 0 - токени питання;
    # 1 - токени контексту.
    # Для відповіді нас цікавлять лише токени контексту.
    sequence_ids = encoded.sequence_ids(0)

    inputs = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    # Модель для extractive QA повертає два набори логітів:
    # start_logits - оцінки для позиції початку відповіді;
    # end_logits - оцінки для позиції кінця відповіді.
    start_logits = outputs.start_logits[0].cpu()
    end_logits = outputs.end_logits[0].cpu()
    # У SQuAD-подібних задачах CLS-токен використовується як варіант "немає відповіді".
    cls_score = start_logits[0] + end_logits[0]

    best_score = -1e9
    best_answer = ""

    # Перебираємо всі можливі позиції початку відповіді.
    for start_idx, start_logit in enumerate(start_logits):
        if sequence_ids[start_idx] != 1:
            continue

        # Перебираємо можливі позиції кінця відповіді.
        for end_idx in range(start_idx, min(start_idx + 30, len(end_logits))):
            if sequence_ids[end_idx] != 1:
                continue

            score = start_logit + end_logits[end_idx]

            if score > best_score:
                start_char = offset_mapping[start_idx][0]
                end_char = offset_mapping[end_idx][1]
                best_score = score
                best_answer = context[start_char:end_char]

    if cls_score > best_score:
        best_answer = ""

    return {
        "answer": best_answer,
        "score": float(best_score),
        "no_answer_score": float(cls_score),
    }

# оцінка натренованої моделі
def evaluate_model(eval_set, model, tokenizer, device):
    model.to(device)
    # завантаження метрики, що використовується для оцінки передбачення
    squad_metric = load("squad_v2")

    # exact - точне співпадіння
    # f1 - середня оцінка F1 прогнозованих токенів відповіді у порівнянні із правильною відповіддю
    # детальніше про метрики: https://huggingface.co/spaces/evaluate-metric/squad_v2
    metrics = {"exact": [], "f1": [], "HasAns_f1": [], "HasAns_exact": [], "NoAns_f1": []}

    for idx, row in enumerate(eval_set):
        answer = answer_question(question=row["question"], context=row["context"], model=model, tokenizer=tokenizer)
        # представлення правильної відповіді
        reference = {"answers": row['answers'], 'id': row['id']}
        # представлення передбаченої відповіді
        prediction = {"id": row['id'], "prediction_text": answer['answer'], "no_answer_probability": 0}
        results = squad_metric.compute(references=[reference], predictions=[prediction])
        # додаємо кожну метрику у список метрик
        for metric_key in metrics.keys():
            if not metric_key in results:
                continue
            metrics[metric_key].append(results[metric_key])

        if idx % 100 == 0:
            print(f"Processed {idx} examples")

    # обчислення середнього значення для кожної метрики
    return {key: sum(val) / len(val) if len(val) > 0 else 0 for key, val in metrics.items()}

evaluate_model(test_rows, model, tokenizer, device)

Processed 0 examples
Processed 100 examples
Processed 200 examples
Processed 300 examples
Processed 400 examples
Processed 500 examples
Processed 600 examples
Processed 700 examples
Processed 800 examples
Processed 900 examples
Processed 1000 examples
Processed 1100 examples
Processed 1200 examples
Processed 1300 examples
Processed 1400 examples


{'exact': 58.75,
 'f1': 72.23755410171982,
 'HasAns_f1': 75.01929886929769,
 'HasAns_exact': 58.04195804195804,
 'NoAns_f1': 61.486486486486484}

## 3. Побудова Retriever-моделі

Для побудови пошукової моделі скористаємося попередньо натренованою моделлю векторних представлень, що для кожної вхідної послідовності формує список чисел. При цьому близькі за сенсом послідовності будуть мати низьке значення для метрик дистанції (або високе для подібності).

[Детальніше про векторні представлення](https://ai.gopubby.com/an-intuitive-101-guide-to-vector-embeddings-ffde295c3558)

[Детальніше про семантичну подібність](https://www.sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html)

[Детальніше про семантичний пошук](https://www.sbert.net/examples/applications/semantic-search/README.html)

[Детальніше про семантичний пошук з FAISS](https://huggingface.co/learn/nlp-course/en/chapter5/6)

In [ ]:
train_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'answers', 'is_impossible'],
        num_rows: 13859
    })
})

In [ ]:
# побудова сховища даних: збір усіх текстів з датасету
docs = list(set([x['context'] for x in train_dataset['train']]))
len(docs), docs[:5]

(2620,
 ['Стиснення аркуша видаляє воду силою; як тільки вода виштовхується з аркуша, для збору води використовується спеціальний вид повсті, який не слід плутати з традиційним; тоді як при виготовленні паперу вручну замість нього використовується промокальний аркуш.',
  "У попередніх сезони шоу критикували через обтяжливий контракт, який учасники повинні були підписати. Нібито компанія 19 Entertainment надмірно контролювала майбутню кар'єру переможців. У тому числі це проявлялося в обов'язку передати керівництву чималу частку від майбутніх заробітків артистів.",
  'корінний алабамєць Аллен Барра різко розкритикував Лі та роман у The Wall Street Journal, назвавши Аттікуса "сховищем епіграм із сухариками", а роман представляє "міф із цукровим покривом" історії Алабами. Барра пише: "Слід перестати робити вигляд, що "Убити пересмішника" - це якась позачасова класика, яка входить до числа великих творів американської літератури. Його безкровний ліберальний гуманізм, на жаль, датується". То

In [ ]:
# завантаження моделі векторного представлення
encoder = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# побудова сховища даних: векторизація усіх текстів та організація їх швидкого пошуку
# за допомогою алгоритмів Facebook AI Similarity Search (FAISS)
db = FAISS.from_texts(texts=docs, embedding=encoder)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

... (2 outputs left out)

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4. ODQA: поєднання Retriever та Reader

In [ ]:
def run_odqa(ranker, reader, query, k=10, handle_impossible_answer=True, threshold=0.5, return_all=False):
    # визначення K релевантних текстів для запиту
    docs_and_scores = ranker.similarity_search_with_score(query, k=k)
    query_answer = None
    answer_candidates = []

    for doc in docs_and_scores:
        context, retrieval_score = doc
        context = context.page_content
        # визначення сегменту-відповіді у кожному релевантному тексті
        # handle_impossible_answer=True - модель може передбачити відсутність відповіді у тексті
        #answer = reader(question=query, context=context, handle_impossible_answer=handle_impossible_answer)
        answer = answer_question(question=query, context=context, model=reader, tokenizer=tokenizer)
        qa_score = answer['score']
        answer_text = answer['answer']
        answer_candidate = (answer_text, qa_score, context)
        if return_all:
            # зберігаємо усі передбачення, щоб повернути K відповідей
            answer_candidates.append(answer_candidate)
        if qa_score < threshold:
            continue
        if query_answer is not None:
            # визначаємо відповідь з найбільшою ймовірністю
            if query_answer[1] >= qa_score or not answer_text:
                continue
        query_answer = answer_candidate
    return query_answer if not return_all else answer_candidates

# оцінка ефективності усього пайплайну: необхідно не лише отримати релевантний текст, але й знайти у ньому правильну відповідь
def eval_odqa(ranker, reader, test_set, k=10, handle_impossible_answer=True, threshold=0.5):
    results = []
    metrics = {"exact": [], "f1": [], "HasAns_f1": [], "HasAns_exact": [], "NoAns_f1": []}
    squad_metric = load("squad_v2")

    for row in test_set:
        query = row['question']
        query_answer = run_odqa(ranker, reader, query, k, handle_impossible_answer, threshold)

        reference = {"answers": row['answers'], 'id': row['id']}
        prediction = {"id": row['id'], "prediction_text": "", "no_answer_probability": 0}
        if query_answer is not None:
            prediction["prediction_text"] = query_answer[0]
        metric_scores = squad_metric.compute(references=[reference], predictions=[prediction])
        for metric_key in metrics.keys():
            if not metric_key in metric_scores:
                continue
            metrics[metric_key].append(metric_scores[metric_key])

        results.append({"predicted": query_answer, "truth": row})
    return results, {key: sum(val) / len(val) if len(val) > 0 else 0 for key, val in metrics.items()}

In [ ]:
queries = [x['question'] for x in test_rows]
random.shuffle(queries)

# передбачення для 10 випадкових запитів
for query in queries[:10]:
    answer = run_odqa(db, model, query, k=10, handle_impossible_answer=True, threshold=0.2)
    if answer:
        print(f"question: {query}\nanswer: {answer[0]}\nscore: {answer[1]}\ncontext: {answer[2]}\n\n")
    else:
        print(f"question: {query}\nanswer: None\nscore: None\ncontext: None\n\n")

question: Що було найважливішими подіями для Гелугу?
answer: 
score: 3.5044939517974854
context: Як каже геолог Стефан Шмід, оскільки в Західних Альпах відбулася метаморфічна подія в Кайнозойській ері, тоді як австральпійські вершини утворилися у Крейдовому періоді, ці дві області демонструють чіткі відмінності у формуванні напів. Поклади флішу в Південних Альпах Ломбардії, ймовірно, траплялися в Крейдовому періоді або пізніше.


question: Яких живих тварин вони утримували в Беренграбені?
answer:  слонів
score: 11.364226341247559
context: Муміфікованого чоловіка, якому було визначено вік 5000 років, було виявлено на льодовику на австрійсько-італійському кордоні в 1991 році. До VI століття до нашої ери кельтська латенська культура добре утвердилася. Ганнібал знаменито перетнув Альпи зі стадом слонів, а римляни мали поселення в цьому регіоні. У 1800 році Наполеон перетнув один з гірських перевалів з 40-тисячною армією. У 18-19 століттях спостерігався приплив натуралістів, письменників та

In [ ]:
# оцінка точності ODQA пайплайну на тестовому наборі
results, metrics = eval_odqa(db, model, test_rows, k=5, threshold=0.0)

In [ ]:
metrics

{'exact': 38.958333333333336,
 'f1': 47.84144381879001,
 'HasAns_f1': 43.8738453663091,
 'HasAns_exact': 32.69230769230769,
 'NoAns_f1': 63.17567567567568}

In [ ]:
results[25:50]

[{'predicted': (' як позитивні, так і негативні',
   6.539314270019531,
   'Елісон Цзін Сю, доцент кафедри менеджменту в Університеті Торонто Скарборо та Апарна Лабру з Північно-Західного університету провели низку досліджень, аналізуючи взаємозв\'язок між освітленням та людськими емоціями. Дослідники попросили учасників оцінити низку речей, таких як: пікантність соусу з курячих крил, агресивність вигаданого персонажа, чюсь привабливість, їхні почуття щодо конкретних слів та смак двох соків - все при різних умовах освітлення. У своєму дослідженні вони виявили, що як позитивні, так і негативні людські емоції відчуваються більш інтенсивно при яскравому світлі. Професор Сю заявив: "ми виявили, що в сонячні дні люди, схильні до депресії, насправді стають ще більш депресивними". Вони також виявили, що тьмяне світло змушує людей приймати більш раціональні рішення та легше врегулювати переговори. У темряві емоції трохи пригнічуються, а при яскравому – посилюються.'),
  'truth': {'id': 'train-

`eval_odqa()` порівнює правильну відповідь лише з найімовірнішою передбаченою, що є справедливим підходом: користувачам важливо отримувати правильний результат на першій позиції. Водночас, важливу роль у оцінюванні пошукових систем можуть відігравати і метрики, що враховують не лише першу позицію, а N передбачень з найбільшими ймовірностями. Наприклад, для двох рішень з приблизно однаковими результатами додаткова оцінка потрапляння правильної відповіді у топ-5 передбачень допоможе визначити рішення з кращою узагальнюючою здібністю.

**Завдання**: модифікуйте функцію `eval_odqa()` так, щоб вона зараховувала правильну відповідь у випадку її знаходження у топ-N (за ймовірністю) передбаченнях. Проведіть повторну оцінку реалізованого рішення та порівняйте результати.

Варіант = Номер_у_списку % 4 + 1

|**Варіант**| N |
|-----------|---|
|**1**      | 3 |
|**2**      | 5 |
|**3**      | 7 |
|**4**      | 10|

In [ ]:
def eval_odqa_first_n(ranker, reader, test_set, k=10, handle_impossible_answer=True, threshold=0.5, top_n=5):
    raise NotImplementedError

## 5. Додаткове завдання. Як поліпшити якість відповідей?

Перегляньте кілька помилок базової системи на валідаційній вибірці. Оберіть одну з них і з’ясуйте, де вона виникла. Чи знайшов retriever уривок із відповіддю? Яку частину цього уривка побачив reader? Яку відповідь система поставила на перше місце?

Запропонуйте й перевірте одну зміну. Наприклад, якщо підозрюєте, що причиною помилок є обрізання довгих текстів, розгляньте валідаційні приклади з відповіддю. У якій частці цих прикладів reader бачить лише частину еталонної відповіді або втрачає її повністю?

Порівняйте базову та змінену системи на тих самих тестових запитаннях за Exact Match і F1 для першої відповіді. Наведіть приклад, який пояснює отриманий результат. Якщо зміна збільшила час пошуку чи обробки запиту, врахуйте це у висновку.